# Predict Seagrass at different times
* Designed to work year by year
* Pick a location and run

In [1]:
import geopandas
import pathlib
import rioxarray
import numpy
import dask.distributed
import pandas
import rasterio

module_path = pathlib.Path.cwd().parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2
import training
import sampling

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:55903/status,
Dashboard: http://127.0.0.1:55903/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:55905,Workers: 0
Dashboard: http://127.0.0.1:55903/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:55925,Total threads: 2
Dashboard: http://127.0.0.1:55926/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:55908,


# Value to edit

In [3]:
all_training_sites =  ["CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau", "Ihutai",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]
large_seagrass_sites =  ["CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Ihutai", "LeftBank_Nov25", "ThePoint_Nov25"]



In [4]:
satellite_classes_all = {'Seagrass': 1, 'Seagrass submerged': 2, 'Gracilaria': 3, 'Gracilaria submerged': 4, 
                           'Ulva': 5, 'Submerged vegetation': 9, 'Microphytobenthos': 10, 'Satmarsh': 14,
                           'Unvegetated': 15, 'Water': 16, 'Terrestrial': 18, 'Rock': 19, 'Mixed': 22,}
satellite_classes_all_unet = {'Seagrass': 0, 'Seagrass submerged': 1, 'Gracilaria': 2, 'Gracilaria submerged': 3, 
                           'Ulva': 4, 'Submerged vegetation': 5, 'Microphytobenthos': 6, 'Satmarsh': 7,
                           'Unvegetated': 8, 'Water': 9, 'Terrestrial': 10, 'Rock': 11, 'Mixed': 12,}

In [5]:
sample_method = "sampling_2"
method_2_threshold = .5
model_max_cloud_cover = 1 # percentage
model_low_tide_delta_hrs = 0
model_low_tide_delta_mins = 30
test_threshold = 0.1

predict_max_cloud_cover = 1 # percentage"
predict_low_tide_delta_hrs = 0
predict_low_tide_delta_mins = 30

train_subset_description = "_train_banks"

model_name = f"test_on_{int(test_threshold*100)}_percent_all_classes{train_subset_description}.joblib"

In [6]:
model_path = utils.get_models_path(sample_method=sample_method, method_2_threshold=method_2_threshold,
                                   low_tide_delta_hrs=model_low_tide_delta_hrs, low_tide_delta_mins=model_low_tide_delta_mins,
                                   max_cloud_cover=model_max_cloud_cover)
model_file = model_path / model_name
prediction_sites = all_training_sites
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [14]:
data_path = utils.get_data_path()
utils.create_data_folders()
predictions_path, satellite_path = utils.get_prediction_path(sample_method, method_2_threshold, model_max_cloud_cover=model_max_cloud_cover, predict_max_cloud_cover=predict_max_cloud_cover,
                                             predict_low_tide_delta_hrs=predict_low_tide_delta_hrs, predict_low_tide_delta_mins=predict_low_tide_delta_mins,
                                             model_low_tide_delta_hrs=model_low_tide_delta_hrs, model_low_tide_delta_mins=model_low_tide_delta_mins)

predictions_path = predictions_path.with_name(f"{predictions_path.name}{train_subset_description}")
predictions_path.mkdir(exist_ok=True, parents=True)

# Cells to run
* Get lowtide satellite images by year and site
* Predict Seagrass across satellite images

### Save satellite
The low-tide no cloud images across sites for a whole year

In [ ]:
for prediction_site in prediction_sites:
    for year in years:
        print(f"{prediction_site}: year {year}")
    
        satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
        site_polygon_file = utils.get_site_polygon_path(prediction_site)
    
        if not satellite_file.exists():
            print(f"\tSave satellite image of the site {prediction_site} around lowtide without cloud in the year {year}")
            site_polygon = geopandas.read_file(site_polygon_file)
            satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                geometry=site_polygon,
                max_cloud_cover=predict_max_cloud_cover,
                year=year,
                low_tide_delta_hrs=predict_low_tide_delta_hrs,
                low_tide_delta_mins=predict_low_tide_delta_mins,
            )
            if len(satellite_data['time']) == 0:
                print("Warning: No satellite data without cloud. Ignore")
            else:
                print("\t\tSave")
                utils.write_netcdf_conventions_in_place(satellite_data)
                satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                utils.write_netcdf_conventions_in_place(satellite_data)
                utils.save_netcdf(satellite_data, satellite_file)
            utils.rgb_from_satellite(satellite_data_path=satellite_file)

### Predict values

In [ ]:
for prediction_site in prediction_sites:
    for year in years:
        print(f"{prediction_site}: year {year}")
        prediction_file = predictions_path / f"{prediction_site}_{year}_model_{model_file.stem}.nc"
        polygon_file = utils.get_site_polygon_path(prediction_site)
        satellite_file = satellite_path / f"{prediction_site}_{year}_sentinel-2.nc"
        
        if not prediction_file.exists():
            model_feature_names_file = model_file.with_name(f"{model_file.stem}_feature_names.csv")
            if not satellite_file.exists():
                print("\tWARNING no satellite file to predict. Rerun the above cell if this is unexpected."
                      " Otherwise this indicates no suitable low-tide and low-cloud dates in the year.")
                continue
            predictions, satellite_data = training.predict_site(test_satellite_file=satellite_file,
                                                                polygon_file=polygon_file, model_file=model_file,
                                                                model_feature_names_file=model_feature_names_file)
            utils.save_netcdf(predictions, prediction_file)
        else:
            predictions = utils.load_classification(filename=prediction_file, chunks=None)
    
        for target_name in ["Seagrass", "Ulva", "Gracilaria"]:
            target_file = predictions_path /  f"{prediction_site}_{year}_model_{model_file.stem}_{target_name.lower()}.gpkg"
            if not target_file.exists():
                target = {"date": [], "geometry": []}
                prediction = utils.load_classification(filename=prediction_file)
                prediction = utils.ensure_grid_indexing(prediction)
                target_id = pandas.read_csv(model_file.with_name(f"{model_file.stem}_class_mappings.csv"), index_col=0)
                target_id = int(target_id.loc[target_id.index == target_name,"satellite_class_id"].squeeze())
                for time_index in range(len(prediction["time"])):
                    mask = predictions.isel(time=time_index) == target_id
                    target["date"].append(str(mask.time.data))
                    geometry = utils.mask_to_polygons(mask).dissolve().geometry
                    target["geometry"].append(geometry[0])
                target = geopandas.GeoDataFrame(target, crs="2193")
                target.to_file(target_file)
                

CatlinsLake: year 2016
	Predict for 1 satellite images
	Predict satellite image for date 2016-04-09
	Combine predictions and write conventions
CatlinsLake: year 2017
	WARNING no satellite file to predict. Rerun the above cell if this is unexpected. Otherwise this indicates no suitable low-tide and low-cloud dates in the year.
CatlinsLake: year 2018
	Predict for 1 satellite images
	Predict satellite image for date 2018-04-02
	Combine predictions and write conventions
CatlinsLake: year 2019
	Predict for 3 satellite images
	Predict satellite image for date 2019-02-21
	Predict satellite image for date 2019-04-07
	Predict satellite image for date 2019-08-17
	Combine predictions and write conventions
CatlinsLake: year 2020
	Predict for 3 satellite images
	Predict satellite image for date 2020-05-08
	Predict satellite image for date 2020-06-22
	Predict satellite image for date 2020-12-02
	Combine predictions and write conventions
CatlinsLake: year 2021
	Predict for 1 satellite images
	Predict